# Easy Cover Maker

This notebook allows you to create AI covers of songs using any RVC v2 model. Simply provide a YouTube URL for the song you want to cover, upload your RVC model, and let the notebook handle the rest. It will download the song, separate the vocals from the instrumentals, convert the vocals using your model, and then remix the track.

## 1. Install Dependencies

This cell will install all the necessary libraries for the cover generator to work. This may take a few minutes.

In [ ]:
#@title Install all the required dependencies
!pip install -q "audio-separator[gpu]"
!pip install -q rvc-python
!pip install -q yt-dlp
!pip install -q gradio
!pip install -q pydub
!pip install -q soundfile

## 2. Import Libraries and Define Helper Functions

In [ ]:
import gradio as gr
import os
import yt_dlp
import torch
import librosa
import soundfile as sf
from pydub import AudioSegment
from audio_separator.separator import Separator
from rvc_python.RVC import RVC

# Create a directory for temporary files
os.makedirs('temp', exist_ok=True)

def download_song(url):
    """Downloads a song from a YouTube URL and saves it as a WAV file."""
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': 'temp/downloaded_song.%(ext)s',
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return 'temp/downloaded_song.wav'

def separate_vocals(audio_path):
    """Separates the vocals from the instrumentals, and then separates the main vocals from the backing vocals."""
    # Initialize the separator
    separator_inst_voc = Separator(output_dir='temp', output_format='wav', log_level='INFO')
    # Separate instrumentals and vocals
    separator_inst_voc.load_model(model_filename='UVR-MDX-NET-Inst-HQ-3.onnx')
    output_paths = separator_inst_voc.separate(audio_path)
    instrumental_path = next((p for p in output_paths if 'Instrumental' in p), None)
    vocals_path = next((p for p in output_paths if 'Vocals' in p), None)

    if not vocals_path:
        raise Exception("Could not separate vocals from the audio.")

    # Separate main and backing vocals
    separator_main_voc = Separator(output_dir='temp', output_format='wav', log_level='INFO')
    separator_main_voc.load_model(model_filename='UVR-MDX-NET-Voc_FT.onnx')
    main_vocals_paths = separator_main_voc.separate(vocals_path)
    main_vocals_path = next((p for p in main_vocals_paths if 'Vocals' in p), None)

    if not main_vocals_path:
        raise Exception("Could not separate main vocals.")

    # Calculate backing vocals by subtracting main vocals from all vocals
    all_vocals_audio, sr = sf.read(vocals_path)
    main_vocals_audio, _ = sf.read(main_vocals_path)
    min_len = min(len(all_vocals_audio), len(main_vocals_audio))
    all_vocals_audio = all_vocals_audio[:min_len]
    main_vocals_audio = main_vocals_audio[:min_len]
    backing_vocals_audio = all_vocals_audio - main_vocals_audio
    backing_vocals_path = 'temp/backing_vocals.wav'
    sf.write(backing_vocals_path, backing_vocals_audio, sr)

    return instrumental_path, main_vocals_path, backing_vocals_path

def convert_vocals(main_vocals_path, backing_vocals_path, rvc_model_path):
    """Converts the main and backing vocals using the RVC model."""
    rvc = RVC(model_path=rvc_model_path, device='cuda' if torch.cuda.is_available() else 'cpu')

    converted_main_vocals_path = 'temp/converted_main_vocals.wav'
    rvc.infer(main_vocals_path, converted_main_vocals_path)

    converted_backing_vocals_path = 'temp/converted_backing_vocals.wav'
    rvc.infer(backing_vocals_path, converted_backing_vocals_path)

    return converted_main_vocals_path, converted_backing_vocals_path

def remix_audio(instrumental_path, converted_main_vocals_path, converted_backing_vocals_path):
    """Remixes the instrumental with the converted vocals."""
    instrumental = AudioSegment.from_wav(instrumental_path)
    main_vocals = AudioSegment.from_wav(converted_main_vocals_path)
    backing_vocals = AudioSegment.from_wav(converted_backing_vocals_path)

    # Lower backing vocals volume
    backing_vocals = backing_vocals - 6  # 6 dB quieter

    # Overlay vocals on instrumental
    remix = instrumental.overlay(main_vocals)
    remix = remix.overlay(backing_vocals)

    output_path = 'cover_song.wav'
    remix.export(output_path, format='wav')
    return output_path

def create_cover(url, rvc_model_file):
    """Main function to create the AI cover."""
    if not url or not rvc_model_file:
        gr.Warning("Please provide a YouTube URL and an RVC model.")
        return None

    try:
        rvc_model_path = rvc_model_file.name
        
        gr.Info("Downloading song...")
        song_path = download_song(url)

        gr.Info("Separating vocals...")
        instrumental_path, main_vocals_path, backing_vocals_path = separate_vocals(song_path)

        gr.Info("Converting vocals...")
        converted_main_vocals_path, converted_backing_vocals_path = convert_vocals(main_vocals_path, backing_vocals_path, rvc_model_path)

        gr.Info("Remixing audio...")
        output_path = remix_audio(instrumental_path, converted_main_vocals_path, converted_backing_vocals_path)

        gr.Success("Cover created successfully!")
        return output_path
    except Exception as e:
        gr.Error(f"An error occurred: {e}")
        return None

## 3. Launch the Gradio Web UI

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Easy Cover Maker")
    gr.Markdown("Enter a YouTube URL, upload your RVC model, and click 'Create Cover' to generate your AI cover song.")
    with gr.Row():
        with gr.Column():
            youtube_url = gr.Textbox(label="YouTube URL")
            rvc_model = gr.File(label="Upload RVC Model (.pth)")
            create_button = gr.Button("Create Cover", variant="primary")
        with gr.Column():
            output_audio = gr.Audio(label="Generated Cover")
            
    create_button.click(
        create_cover,
        inputs=[youtube_url, rvc_model],
        outputs=output_audio
    )

demo.launch(debug=True, share=True)